# Recorte de enunciados de PDFs de emestrada — paso a paso

In [1]:
# %pip install pdfplumber pypdf pandas

import re
import os
import shutil
from pathlib import Path
import unicodedata

import pandas as pd
import pdfplumber
from pypdf import PdfReader, PdfWriter, Transformation
from IPython.display import display

In [2]:
root = "/home/daniel/code/clasificador_selectividad"
os.chdir(root)

In [3]:
ANCHO_MIN, ALTO_MIN = 300, 20   # tamaño mínimo para considerar un rectángulo como recuadro de enunciado
MARGEN = 2                      # holgura alrededor del borde al recortar
DPI = 200                       # resolución de los PNG
A4 = (595.28, 841.89)
MARGEN_A4 = 40                  # margen de la hoja de salida
SEPARACION = 15                 # hueco entre recuadros apilados

In [4]:
def detectar_recuadros(page):
    vistos = {}
    for r in page.rects:
        if r["width"] > ANCHO_MIN and r["height"] > ALTO_MIN:
            clave = tuple(round(r[k]) for k in ("x0", "top", "x1", "bottom"))
            vistos[clave] = r
    return sorted(vistos.values(), key=lambda r: r["top"])

In [5]:
# PATRON = re.compile(
#     r"""
#     (?P<asig>SOCIALES[ \t]+(?:II|2)|MATEM[AÁ]TICAS[ \t]+(?:II|2)|QU[IÍ]MICA|F[IÍ]SICA)
#     [ \t.,:;-]*
#     (?P<anio>(?:19|20)\d{2})
#     [ \t.,:;-]*
#     (?P<conv>(?:JUNIO|JULIO|SEPTIEMBRE|PONENCIA|RESERVA|MODELO)(?:[ \t]*\d+)?)
#     [ \t.,:;-]*
#     (?P<ej>[^\n\r]*?)
#     [ \t.]*\r?$
#     """,
#     re.IGNORECASE | re.MULTILINE | re.VERBOSE,
# )

In [6]:
def process_metadata(metadata):
    metadata = metadata.lower()
    asignaturas = {
        "química" : "Química",
        "física" : "Física",
        "sociales ii" : "Mates CCSS",
        "matemáticas ii" : "Matemáticas II"
    }

    # extraer asignatura
    for k,v in asignaturas.items():
        if k in metadata:
            asignatura = v
            break
        else:
            asignatura = "asignatura_desconocida"

    # extraer año
    patron = r"2\d{3}"
    year = re.findall(patron, metadata)[0]

    # extraer convocatoria y ejercicio
    convocatorias = ["junio", "julio", "septiembre",
                     "modelo", "ponencia",
                     "reserva"]
    for conv in convocatorias[:-2]:
        if conv in metadata:
            convocatoria = conv.title()
            convocatoria_position = metadata.find(conv)
            ejercicio = (" ".join(metadata[convocatoria_position:].split(" ")[1:])
                         .title()
            )
            break

    if "reserva" in metadata:
        convocatoria_position = metadata.find("reserva")
        convocatoria = (" ".join(metadata[convocatoria_position:]
                                 .split(" ")[:2])
                                 .title()
                                 .replace(".", "")
        )
        ejercicio = (" ".join(
            metadata[convocatoria_position:].split(" ")[2:]
            ).title()
        )


    return (asignatura, year, convocatoria, ejercicio)
    


In [7]:
def slug(texto):
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return re.sub(r"\W+", "_", texto).strip("_").lower()

#print(slug("RESERVA 1"), "|", slug("SEPTIEMBRE"), "|", slug("2023 - Integrales"))

In [8]:
def procesar_pdf(ruta_pdf, output_folder, png=True, dpi=DPI, uno_por_pagina=False):
    filas, recortes = [], []
    png_folder = output_folder / "enunciados_PNG" / f"{ruta_pdf.stem}_png"

    # 1) Detección con pdfplumber
    with pdfplumber.open(ruta_pdf) as pdf:
        for n, page in enumerate(pdf.pages):
            cajas = detectar_recuadros(page)
            for k, r in enumerate(cajas, start=1):
                x0 = max(r["x0"] - MARGEN, 0)
                x1 = min(r["x1"] + MARGEN, page.width)
                top = max(r["top"] - MARGEN, 0)
                bottom = min(r["bottom"] + MARGEN, page.height)

                recorte = page.crop((x0, top, x1, bottom))
                texto_raw = recorte.extract_text() or ""
                lineas = texto_raw.split("\n")
                texto = "\n".join(lineas[:-2])
                metadata = lineas[-1]
                asignatura, anio, conv, ej = process_metadata(metadata)
                # texto = texto_raw.replace("\n", " ")
                # metadata = PATRON.search(texto_raw)   # buscar con saltos de línea

                if metadata:
                    ej_slug = slug(ej)
                    nombre = f"{anio}_{slug(conv)}_{ej_slug}"
                    # ej_slug = slug(re.sub(r"EJERCICIO", "", metadata["ej"], flags=re.IGNORECASE))
                    # nombre = f"{metadata['anio']}_{slug(metadata['conv'])}_{ej_slug}"
                else:
                    nombre = f"{slug(ruta_pdf.stem)}_p{n + 1:02d}"

                imagen = ""
                if png:
                    png_folder.mkdir(parents=True, exist_ok=True)
                    destino = png_folder / f"{nombre}.png"
                    i = 2
                    while destino.exists():
                        destino = png_folder / f"{nombre}_{i}.png"
                        i += 1
                    recorte.to_image(resolution=dpi).save(destino)
                    imagen = str(destino.relative_to(output_folder))

                off_x, off_y = float(page.bbox[0]), float(page.bbox[1])
                recortes.append((n, off_x + x0, off_y + page.height - bottom,
                                 off_x + x1, off_y + page.height - top))

                aviso = ""
                if not metadata:
                    aviso = "sin metadatos"
                elif len(cajas) > 1:
                    aviso = f"{len(cajas)} recuadros en la página"

                # asignatura y tema
                #asignatura = ruta_pdf.parts[-3].replace(" - Temas", "").replace(" ", "").lower()
                tema = ruta_pdf.stem.split(" - ")[1]

                filas.append({
                    "asignatura": asignatura,
                    "año" : anio,
                    "convocatoria" : conv,
                    "ejercicio" : ej,
                    "tema": tema,
                    "metadata" : metadata,
                    "enunciado": texto,
                    "archivo": ruta_pdf.name,
                    "imagen": imagen,
                    "aviso": aviso,
                    #"año": metadata["anio"] if metadata else "",
                    #"convocatoria": metadata["conv"].title() if metadata else "",
                    #"ejercicio": metadata["ej"] if metadata else "",
                })

    if not recortes:
        return pd.DataFrame(filas), "ningún recuadro detectado"
    else:
        return pd.DataFrame(filas), ""

In [9]:
# RUTA_PDF = Path("./pdfplumber/solubilidad/2024 - Solubilidad.pdf")
# #CARPETA_ENTRADA = Path("./pdf_recortados")
# CARPETA_SALIDA = Path("./pdf_recortados")

# CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
# print("¿Existe el PDF de prueba?", RUTA_PDF.exists())

In [10]:
# filas, aviso = procesar_pdf(RUTA_PDF, CARPETA_SALIDA)
# print("Aviso:", aviso or "ninguno")
# pd.DataFrame(filas)

## Procesar carpeta

In [20]:
def process_folder(folder, files_to_process_limit=None):

    # '/home/daniel/code/clasificador_selectividad/source_pdf_files/Física - Temas/campo_gravitatorio'
    source_folder = Path(folder)
    output_folder_name = "/".join(source_folder.parts[-2:])
    output_folder = Path(f"{root}/enunciados_solo/{output_folder_name}")
    output_folder.mkdir(parents=True, exist_ok=True)
    print(f"Procesando carpeta: {output_folder_name}")

    pdf_files = sorted(p for p in source_folder.rglob("*.pdf") if not p.stem.endswith("_enunciados"))
    if files_to_process_limit is not None:
        pdf_files = pdf_files[:files_to_process_limit]

    print(f"{len(pdf_files)} PDFs a procesar\n")

    dataframes, problemas = [], []
    for index, file_path in enumerate(pdf_files, start=1):
        try:
            df, aviso = procesar_pdf(file_path, output_folder)
        except Exception as e:
            problemas.append({"pdf": str(file_path), "problema": f"ERROR: {e}"})
            print(f"[{index}/{len(pdf_files)}] ERROR  {file_path.name}: {e}")
            continue

        dataframes.append(df)

        if aviso:
            problemas.append({"pdf": str(file_path), "problema": aviso})
            print(f"[{index}/{len(pdf_files)}] AVISO  {file_path.name}: {aviso}")
        else:
            print(f"[{index}/{len(pdf_files)}] OK     {file_path.name}: {len(df)} enunciados")

    # Guardar resultados en CSV
    enunciados_df = pd.DataFrame()
    for d in dataframes:
        enunciados_df = pd.concat([enunciados_df, d])
            
    enunciados_df.to_csv(output_folder / f"{output_folder.name}.csv", index=False)

    if problemas:
        problemas_df = pd.DataFrame(problemas)
        problemas_df.to_csv(output_folder / f"{output_folder.name}_problemas.csv",
                            index=False, encoding="utf-8")

    print("*" * 40)
    print(f"\nProcesamiento completado. Resultados guardados en: {output_folder}\n\n")

## Procesar carpetas

Normalizar texto de nombres de temas y carpetas

In [12]:
def normalize_text(texto):
    descompuesto = unicodedata.normalize("NFD", texto)
    return "".join(c for c in descompuesto if not unicodedata.combining(c))

Obtener las rutas de los archivos

In [13]:
os.chdir(f"{root}/source_pdf_files")

In [14]:
carpetas_asignaturas = {d.stem:[] for d in Path(".").iterdir() if d.is_dir()}

In [15]:
for asignatura in carpetas_asignaturas:
    carpetas_asignaturas[asignatura] = [d.stem for d in Path(f"./{asignatura}").iterdir() if d.is_dir()]

In [16]:
carpetas_asignaturas

{'QUÍMICA - TEMAS': ['Termoquímica',
  'Reactividad Orgánica',
  'Ácido Base',
  'Enlace Químico',
  'Cantidad Química',
  'REDOX',
  'Conf',
  'Solubilidad',
  'Equilibrio Químico',
  'Formulación'],
 'MATES II - TEMAS': ['Sistemas de Ecuaciones Lineales',
  'Probabilidad',
  'Distribución normal y binomial',
  'Matrices y Determinantes',
  'Funciones',
  'Geometría',
  'Integrales'],
 'FÍSICA - TEMAS': ['Movimiento Ondulatorio',
  'Óptica geométrica',
  'Física cuántica y nuclear',
  'Campo gravitatorio',
  'Campo eléctrico y magnético'],
 'MATES CCSS - TEMAS': ['Programación Lineal',
  'Distribución Binomial',
  'Probabilidad',
  'Funciones CCSS',
  'Matrices y Determinantes',
  'Sistemas de Ecuaciones Lineales CCSS',
  'Contraste de Hipótesis',
  'Inferencia Estadística']}

In [17]:
folder_paths = []
for asignatura, carpetas in carpetas_asignaturas.items():
    for carpeta in carpetas:
        folder_paths.append(f"{root}/source_pdf_files/{asignatura}/{carpeta}")

In [18]:
folder_paths[:5]

['/home/daniel/code/clasificador_selectividad/source_pdf_files/QUÍMICA - TEMAS/Termoquímica',
 '/home/daniel/code/clasificador_selectividad/source_pdf_files/QUÍMICA - TEMAS/Reactividad Orgánica',
 '/home/daniel/code/clasificador_selectividad/source_pdf_files/QUÍMICA - TEMAS/Ácido Base',
 '/home/daniel/code/clasificador_selectividad/source_pdf_files/QUÍMICA - TEMAS/Enlace Químico',
 '/home/daniel/code/clasificador_selectividad/source_pdf_files/QUÍMICA - TEMAS/Cantidad Química']

# PROCESAR LOS ARCHIVOS

In [21]:
for folder in folder_paths:
    folder_path = Path(folder)
    process_folder(folder, files_to_process_limit=None)  # Limitar a 5 archivos por carpeta para pruebas

Procesando carpeta: QUÍMICA - TEMAS/Termoquímica
19 PDFs a procesar

[1/19] OK     2000 - Termoquímica.pdf: 5 enunciados
[2/19] OK     2001 - Termoquímica.pdf: 7 enunciados
[3/19] OK     2002 - Termoquímica.pdf: 6 enunciados
[4/19] OK     2003 - Termoquímica.pdf: 7 enunciados
[5/19] OK     2004 - Termoquímica.pdf: 7 enunciados
[6/19] OK     2005 - Termoquímica.pdf: 7 enunciados
[7/19] OK     2006 - Termoquímica.pdf: 7 enunciados
[8/19] OK     2007 - Termoquímica.pdf: 7 enunciados
[9/19] OK     2008 - Termoquímica.pdf: 7 enunciados
[10/19] OK     2009 - Termoquímica.pdf: 7 enunciados
[11/19] OK     2010 - Termoquímica.pdf: 7 enunciados
[12/19] OK     2011 - Termoquímica.pdf: 7 enunciados
[13/19] OK     2012 - Termoquímica.pdf: 8 enunciados
[14/19] OK     2013 - Termoquímica.pdf: 6 enunciados
[15/19] OK     2014 - Termoquímica.pdf: 7 enunciados
[16/19] OK     2015 - Termoquímica.pdf: 6 enunciados
[17/19] OK     2016 - Termoquímica.pdf: 6 enunciados
[18/19] OK     2024 - Termoquímica.pdf:

KeyboardInterrupt: 

## Mover archivos .csv

In [ ]:
os.chdir(root)

In [ ]:
Path("csv_enunciados").mkdir(exist_ok=True)

In [ ]:
os.chdir("cropped_pdf_files")

In [ ]:
for folder in Path(".").iterdir():
    if folder.is_dir():
        subject = folder.name.split(" - ")[0].replace(" ", "_").lower()
        csv_files = list(folder.rglob("*.csv"))
        for csv_file in csv_files:
            new_filename = f"{subject}_{csv_file.stem}.csv"
            if "_problemas" in csv_file.name:
                Path(f"{root}/csv_enunciados/problemas").mkdir(exist_ok=True)
                output_csv_path = Path(f"{root}/csv_enunciados/problemas/{subject}_{csv_file.stem}_problemas.csv")
            else:
                new_filename = f"{subject}_{csv_file.stem}_enunciados.csv"
                output_csv_path = Path(f"{root}/csv_enunciados/{new_filename}")

            shutil.move(csv_file, output_csv_path)